# 02. Master Data Preprocessing

## Objective
This notebook cleans and standardizes the project master dataset used to enrich operational data with business and technical project attributes.

## Input
- Raw master dataset (`master_bronze.csv`)

## Output
- Cleaned master dataset for downstream integration and analysis.

In [225]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading
This section loads the consolidated dataset used for analysis.

In [226]:
file_path = "../data/bronze/master_bronze.csv"
df_master = pd.read_csv(file_path, sep=";", encoding="utf-8")
df_master.head()

,ID_Proyecto,Nombre Proyecto,Razon Social Cliente,Contacto Cliente,Contrato,Plazo Años,Entrega Equipos,Tarifa o descuento,Descuento %%,Tarifa Inicial COP/kWh,...,% Ahorro Proyectado energia,Fecha Firma de Contrato,FPO,Fecha de energizacion,Fecha ultimo mantenimiento,Consumo anterior al sistema (KWh),Generacion mensual de la propuesta,Ahorro Mensual Cotizacion,HSP,PR PVsyst
0,EVT-2021-0001,La Merced Ipiales,INELCO .S.A.S.,3185309470,EPC,NaN,NaN,NaN,NaN,NA,...,71%,23/03/2021,14/06/2022,9/02/2022,7/07/2024,8500,5509,NaN,"3,9",80%
1,EVT-2021-0002,Seguridad del Sur,SEGURIDAD DEL SUR LIMITADA,3046581766,EPC,NaN,NaN,NaN,NaN,NA,...,94%,9/07/2021,6/11/2021,8/10/2021,23/03/2023,4045,4000,NaN,"3,8",80%
2,EVT-2021-0003,Dispropan Pasto,INVEESE S.A.S,3137501606,EPC,NaN,NaN,Tarifa,NaN,"$ 713,00",...,88%,23/09/2021,7/03/2022,17/02/2022,24/06/2024,3161,3200,NaN,"4,1",80%
3,EVT-2022-0004,La Merced Aurora,INVEESE S.A.S,3183659405,EPC,NaN,NaN,NaN,NaN,NA,...,28%,28/01/2022,29/10/2022,11/07/2022,17/06/2024,23000,6314,NaN,"4,09",80%
4,EVT-2022-0005,Casa Woodcock,HENRY WOODCOCK DELGADO,NaN,EPC,NaN,NaN,NaN,NaN,NA,...,NaN,7/02/2022,13/03/2022,11/02/2022,18/10/2023,965,216,NaN,"4,1",80%


## Data Exploration

Initial inspection of dataset structure, data types, and data quality issues.

In [227]:
df_master.shape

(105, 32)

In [228]:
df_master.info()

<class 'pandas.DataFrame'>
RangeIndex: 105 entries, 0 to 104
Data columns (total 32 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   ID_Proyecto                          105 non-null    str    
 1   Nombre Proyecto                      105 non-null    str    
 2   Razon Social Cliente                 105 non-null    str    
 3   Contacto Cliente                     82 non-null     str    
 4   Contrato                             105 non-null    str    
 5   Plazo Años                           37 non-null     float64
 6   Entrega Equipos                      37 non-null     str    
 7   Tarifa o descuento                   37 non-null     str    
 8    Descuento        %%                 24 non-null     str    
 9   Tarifa Inicial  COP/kWh              105 non-null    str    
 10  Tipo                                 105 non-null    str    
 11  Tipo 2                               105 no

## Data Cleaning

Handling duplicates and missing values.

In [229]:
df_master.isnull().sum()

ID_Proyecto                             0
Nombre Proyecto                         0
Razon Social Cliente                    0
Contacto Cliente                       23
Contrato                                0
Plazo Años                             68
Entrega Equipos                        68
Tarifa o descuento                     68
 Descuento        %%                   81
Tarifa Inicial  COP/kWh                 0
Tipo                                    0
Tipo 2                                  0
Potencia instalada (KW)                 0
Cantidad de paneles                     0
Modelo de paneles                       0
Potencia Panel                          0
Cantidad inversor                       0
Modelo de inversores                    0
Potencia inversores (KW)                0
Area (m2)                               1
Ciudad                                  0
Departamento                            0
% Ahorro Proyectado energia             5
Fecha Firma de Contrato           

In [230]:
(df_master.isnull().sum() / len(df_master)) * 100

ID_Proyecto                             0.000000
Nombre Proyecto                         0.000000
Razon Social Cliente                    0.000000
Contacto Cliente                       21.904762
Contrato                                0.000000
Plazo Años                             64.761905
Entrega Equipos                        64.761905
Tarifa o descuento                     64.761905
 Descuento        %%                   77.142857
Tarifa Inicial  COP/kWh                 0.000000
Tipo                                    0.000000
Tipo 2                                  0.000000
Potencia instalada (KW)                 0.000000
Cantidad de paneles                     0.000000
Modelo de paneles                       0.000000
Potencia Panel                          0.000000
Cantidad inversor                       0.000000
Modelo de inversores                    0.000000
Potencia inversores (KW)                0.000000
Area (m2)                               0.952381
Ciudad              

In [231]:
cols_to_drop = [col for col in df_master.columns if "Unnamed" in col]
df_master = df_master.drop(columns=cols_to_drop)

In [232]:
df_master.columns = (
    df_master.columns
    .str.strip()
    .str.lower()
    .str.replace("á", "a", regex=False)
    .str.replace("é", "e", regex=False)
    .str.replace("í", "i", regex=False)
    .str.replace("ó", "o", regex=False)
    .str.replace("ú", "u", regex=False)
    .str.replace("ñ", "n", regex=False)
    .str.replace(" ", "_", regex=False)
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.replace("%", "porcentaje", regex=False)
    .str.replace("/", "_", regex=False)
)

In [233]:
def clean_numeric_column(series, kind="numeric", divide_by_100=False):
    s = series.astype(str).str.strip()

    s = s.replace({
        "None": np.nan,
        "nan": np.nan,
        "NaN": np.nan,
        "": np.nan
    })

    if kind == "percentage":
        s = s.str.replace("%", "", regex=False)
    elif kind == "currency":
        s = s.str.replace("$", "", regex=False)

    s = s.str.replace(" ", "", regex=False)
    s = s.str.replace(",", ".", regex=False)
    s = s.str.replace(r"[^0-9\.\-]", "", regex=True)

    s = pd.to_numeric(s, errors="coerce")

    if divide_by_100:
        s = s / 100

    return s

In [234]:
df_master = df_master.rename(columns={
    "id_proyecto": "id_proyecto",
    "descuento________porcentajeporcentaje": "descuento_porcentaje",
    "tarifa_inicial__cop_kwh": "tarifa_inicial_cop_kwh"
})

In [235]:
df_master.columns.tolist()

['id_proyecto',
 'nombre_proyecto',
 'razon_social_cliente',
 'contacto_cliente',
 'contrato',
 'plazo_anos',
 'entrega_equipos',
 'tarifa_o_descuento',
 'descuento_porcentaje',
 'tarifa_inicial_cop_kwh',
 'tipo',
 'tipo_2',
 'potencia_instalada_kw',
 'cantidad_de_paneles',
 'modelo_de_paneles',
 'potencia_panel',
 'cantidad_inversor',
 'modelo_de_inversores',
 'potencia_inversores_kw',
 'area_m2',
 'ciudad',
 'departamento',
 'porcentaje_ahorro_proyectado_energia',
 'fecha_firma_de_contrato',
 'fpo',
 'fecha_de_energizacion',
 'fecha_ultimo_mantenimiento',
 'consumo_anterior_al_sistema_kwh',
 'generacion_mensual_de_la_propuesta',
 'ahorro_mensual_cotizacion',
 'hsp',
 'pr_pvsyst']

In [236]:
df_master["departamento"] = df_master["departamento"].replace({
    "NariÃ±o": "Nariño"
})

In [237]:
df_master["fpo_original"] = df_master["fpo"]
df_master["fecha_ultimo_mantenimiento_original"] = df_master["fecha_ultimo_mantenimiento"]

In [238]:
def clean_numeric_column(series, kind="numeric", divide_by_100=False):
    s = series.astype(str).str.strip()

    # Null-like text only
    s = s.replace({
        "None": np.nan,
        "nan": np.nan,
        "NaN": np.nan,
        "": np.nan
    })

    # Remove symbols depending on type
    if kind == "percentage":
        s = s.str.replace("%", "", regex=False)
    elif kind == "currency":
        s = s.str.replace("$", "", regex=False)

    # Normalize spaces and decimal separator
    s = s.str.replace(" ", "", regex=False)
    s = s.str.replace(",", ".", regex=False)

    # Keep only digits, decimal point and minus sign
    s = s.str.replace(r"[^0-9\.\-]", "", regex=True)

    s = pd.to_numeric(s, errors="coerce")

    if divide_by_100:
        s = s / 100

    return s

In [239]:
percentage_cols = [
    "pr_pvsyst",
    "descuento_porcentaje",
    "porcentaje_ahorro_proyectado_energia"
]

for col in percentage_cols:
    if col in df_master.columns:
        df_master[col] = clean_numeric_column(df_master[col], kind="percentage", divide_by_100=True)

In [240]:
currency_cols = [
    "tarifa_inicial_cop_kwh",
    "ahorro_mensual_cotizacion"
]

for col in currency_cols:
    if col in df_master.columns:
        df_master[col] = clean_numeric_column(df_master[col], kind="currency")

In [241]:
numeric_cols = [
    "plazo_anos",
    "potencia_instalada_kw",
    "cantidad_de_paneles",
    "potencia_panel",
    "cantidad_inversor",
    "potencia_inversores_kw",
    "area_m2",
    "porcentaje_ahorro_proyectado_energia",
    "consumo_anterior_al_sistema_kwh",
    "generacion_mensual_de_la_propuesta",
    "hsp"
]

for col in numeric_cols:
    if col in df_master.columns:
        df_master[col] = clean_numeric_column(df_master[col], kind="numeric")

In [242]:
date_cols = [
    "fecha_firma_de_contrato",
    "fpo",
    "fecha_energizacion",
    "fecha_ultimo_mantenimiento",
    "entrega_equipos"
]

for col in date_cols:
    if col in df_master.columns:
        df_master[col] = pd.to_datetime(df_master[col], dayfirst=True, errors="coerce")

C:\Users\isaro\AppData\Local\Temp\ipykernel_10960\1068227983.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_master[col] = pd.to_datetime(df_master[col], dayfirst=True, errors="coerce")


In [243]:
df_master[[
    "id_proyecto",
    "potencia_instalada_kw",
    "hsp",
    "pr_pvsyst",
    "descuento_porcentaje",
    "tarifa_inicial_cop_kwh",
    "fpo",
    "fecha_ultimo_mantenimiento",
]].head(10)

,id_proyecto,potencia_instalada_kw,hsp,pr_pvsyst,descuento_porcentaje,tarifa_inicial_cop_kwh,fpo,fecha_ultimo_mantenimiento
0,EVT-2021-0001,58.800,3.90,0.8,NaN,NaN,2022-06-14,2024-07-07
1,EVT-2021-0002,38.520,3.80,0.8,NaN,NaN,2021-11-06,2023-03-23
2,EVT-2021-0003,28.620,4.10,0.8,NaN,713.0,2022-03-07,2024-06-24
3,EVT-2022-0004,65.400,4.09,0.8,NaN,NaN,2022-10-29,2024-06-17
4,EVT-2022-0005,2.250,4.10,0.8,NaN,NaN,2022-03-13,2023-10-18
5,EVT-2022-0006,93.150,3.91,0.8,NaN,NaN,2022-04-30,2023-04-13
6,EVT-2022-0007,3.820,4.10,0.8,NaN,NaN,2023-06-01,2024-12-05
7,EVT-2022-0008,7.630,4.50,0.8,NaN,NaN,2022-08-09,2025-01-08
8,EVT-2022-0009,4.887,4.10,0.8,NaN,NaN,2022-09-01,2024-07-18
9,EVT-2022-0010,128.400,4.10,0.8,NaN,NaN,2022-05-03,2024-05-06


In [244]:
df_master[[
    "potencia_instalada_kw",
    "hsp",
    "pr_pvsyst",
    "descuento_porcentaje",
    "tarifa_inicial_cop_kwh"
]].isnull().sum()

potencia_instalada_kw      4
hsp                        0
pr_pvsyst                  0
descuento_porcentaje      81
tarifa_inicial_cop_kwh    85
dtype: int64

In [245]:
df_master["fpo"] = pd.to_datetime(df_master["fpo"], dayfirst=True, errors="coerce")
df_master["fecha_firma_de_contrato"] = pd.to_datetime(df_master["fecha_firma_de_contrato"], dayfirst=True, errors="coerce")
df_master["fecha_energizacion"] = pd.to_datetime(df_master["fecha_de_energizacion"], dayfirst=True, errors="coerce")
df_master["fecha_ultimo_mantenimiento"] = pd.to_datetime(df_master["fecha_ultimo_mantenimiento"], dayfirst=True, errors="coerce")

In [246]:
numeric_cols = [
    "potencia_instalada_kw",
    "hsp",
    "pr_pvsyst",
    "plazo_anos",
    "descuento",
    "tarifa_inicial_cop_kwh"
]

for col in numeric_cols:
    if col in df_master.columns and df_master[col].dtype == "object":
        df_master[col] = df_master[col].astype(str).str.replace(",", ".", regex=False)
    if col in df_master.columns:
        df_master[col] = pd.to_numeric(df_master[col], errors="coerce")

## Project Status Normalization

The `fpo` and maintenance-related fields contain mixed data types, including valid dates and categorical values such as "limitado", "EC" (en curso), and "NR" (no reportado).

To preserve this information, the original values were stored and used to create new categorical status variables. This allows distinguishing between operational states (e.g., limited operation, in progress, missing data) and valid timestamps.

This step ensures that critical business context is not lost during datetime conversion and enables more accurate downstream analysis.

In [247]:
df_master["fpo_estado"] = np.select(
    [
        df_master["fpo_original"].eq("limitado"),
        df_master["fpo_original"].eq("EC"),
        df_master["fpo_original"].eq("NR"),
        df_master["fpo"].isna()
    ],
    [
        "limitado",
        "en_curso",
        "no_reportado",
        "no_fecha"
    ],
    default="fecha_valida"
)

In [248]:
df_master["mantenimiento_estado"] = np.select(
    [
        df_master["fecha_ultimo_mantenimiento_original"].eq("NR"),
        df_master["fecha_ultimo_mantenimiento"].isna()
    ],
    [
        "no_reportado",
        "no_fecha"
    ],
    default="fecha_valida"
)

In [249]:
df_master["estado_operativo"] = np.select(
    [
        df_master["fpo_estado"].eq("limitado"),
        df_master["fpo_estado"].eq("EC"),
        df_master["fpo_estado"].eq("NR"),
        df_master["fpo_estado"].eq("fecha_valida")
    ],
    [
        "limitado",
        "en_curso",
        "no_reportado",
        "en_marcha"
    ],
    default="desconocido"
)

In [250]:
df_master

,id_proyecto,nombre_proyecto,razon_social_cliente,contacto_cliente,contrato,plazo_anos,entrega_equipos,tarifa_o_descuento,descuento_porcentaje,tarifa_inicial_cop_kwh,...,generacion_mensual_de_la_propuesta,ahorro_mensual_cotizacion,hsp,pr_pvsyst,fpo_original,fecha_ultimo_mantenimiento_original,fecha_energizacion,fpo_estado,mantenimiento_estado,estado_operativo
0,EVT-2021-0001,La Merced Ipiales,INELCO .S.A.S.,3185309470,EPC,NaN,NaT,NaN,NaN,NaN,...,5509.0,NaN,3.90,0.8,14/06/2022,7/07/2024,2022-02-09,fecha_valida,fecha_valida,en_marcha
1,EVT-2021-0002,Seguridad del Sur,SEGURIDAD DEL SUR LIMITADA,3046581766,EPC,NaN,NaT,NaN,NaN,NaN,...,4000.0,NaN,3.80,0.8,6/11/2021,23/03/2023,2021-10-08,fecha_valida,fecha_valida,en_marcha
2,EVT-2021-0003,Dispropan Pasto,INVEESE S.A.S,3137501606,EPC,NaN,NaT,Tarifa,NaN,713.00,...,3200.0,NaN,4.10,0.8,7/03/2022,24/06/2024,2022-02-17,fecha_valida,fecha_valida,en_marcha
3,EVT-2022-0004,La Merced Aurora,INVEESE S.A.S,3183659405,EPC,NaN,NaT,NaN,NaN,NaN,...,6314.0,NaN,4.09,0.8,29/10/2022,17/06/2024,2022-07-11,fecha_valida,fecha_valida,en_marcha
4,EVT-2022-0005,Casa Woodcock,HENRY WOODCOCK DELGADO,NaN,EPC,NaN,NaT,NaN,NaN,NaN,...,216.0,NaN,4.10,0.8,13/03/2022,18/10/2023,2022-02-11,fecha_valida,fecha_valida,en_marcha
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,EVT-2025-0101,Surtisur 2025,DISTRIBUIDORA SURTISUR DE NARIÑO SAS,3207185534,EPC,NaN,NaT,NaN,NaN,842.44,...,5298.0,NaN,4.10,0.8,NaN,NaN,NaT,no_fecha,no_fecha,desconocido
101,EVT-2025-0102,Conjunto la Estacion,CONJUNTO RESIDENCIAL LA ESTACION,3175123529,PPA,10.0,NaT,si,0.35,920.00,...,8684.0,NaN,4.10,0.8,NaN,NaN,NaT,no_fecha,no_fecha,desconocido
102,EVT-2025-0103,Club Campestre Palmira,CLUB CAMPESTRE DE PALMIRA CORPORACION,3155721163,EPC,NaN,NaT,NaN,NaN,650.73,...,11492.0,NaN,4.70,0.8,NaN,NaN,NaT,no_fecha,no_fecha,desconocido
103,EVT-2025-0104,CC Valle de Atriz 1,CENTRO EMPRESARIAL VALLE DE ATRIZ PROPIEDAD HO...,3104253047,EPC,NaN,NaT,NaN,NaN,771.64,...,148667.0,NaN,4.10,0.8,NaN,NaN,NaT,no_fecha,no_fecha,desconocido


In [251]:
df_master.to_csv("../data/silver/master_silver.csv", index=False)